# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 26.3 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 10.5 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Zero Shot

In [ ]:
import pandas as pd
data = pd.read_excel('Biology-QA-ArabicPrompt-Zero Shot.xlsx')

In [ ]:
data.shape

(200, 1)

# Predict

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data))):
    prompt = data.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [02:50<00:00,  1.17it/s]


In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
,5
د) أول إجابتين,4
أ / ب / ج / د) الإجابة,4
أ) التمهيدي,2
ج) العبارتان صحيحتان,2
...,...
ب) 6,1
ب) عدد الثغور قليل وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية,1
أ) العدد,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_zero['Normalized Prediction'] = nor_pre

In [ ]:
pred_zero['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,79
A,61
C,34
D,17
Unclassified,9


In [ ]:
pred_zero['prompt'] = data['prompt']
pred_zero.head()

,Predicted,Normalized Prediction,prompt
0,د) 36,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,ب) الخلايا وحيدة النواة,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,ب) الشرايين,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,أ) 8,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,د) اصماغ,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_zero.to_excel('Allam Zero Shot Biology Math.xlsx', index = False)

In [ ]:
true = pd.read_excel('sample_biology.xlsx')
y_true = true['Answer Key'].values
print(classification_report(y_true, pred_zero['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3115    0.5278    0.3918        36
           B     0.3544    0.4746    0.4058        59
           C     0.5000    0.2742    0.3542        62
           D     0.5294    0.2093    0.3000        43
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.3650       200
   macro avg     0.3391    0.2972    0.2903       200
weighted avg     0.4294    0.3650    0.3645       200



# Pred Few Shot

In [ ]:
data2 = pd.read_excel('Biology-QA-ArabicPrompt-Few Shot.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2))):
    prompt = data2.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الإجابة:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [03:36<00:00,  1.08s/it]


In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = pred
pred_few['Predicted'].value_counts()

,count
Predicted,
د) أول إجابتين,4
ج) العبارتان صحيحتان,3
د) آخر إجابتين,3
د) 36,2
ج) 14,2
...,...
أ) الهرمونات,1
ب) 6,1
ب) عدد الثغور قليل وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية كبير,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "أ)" in pr:
    nor_pre.append("A")
  elif "أ / أ" in pr:
    nor_pre.append("A")
  elif "أ / ب / ج / د)" in pr:
    nor_pre.append("Unclassified")
  elif "ب)" in pr:
    nor_pre.append("B")
  elif "ب" in pr:
    nor_pre.append("B")
  elif "ج)" in pr:
    nor_pre.append("C")
  elif "ج" in pr:
    nor_pre.append("C")
  elif "د)" in pr:
    nor_pre.append("D")
  else:
    print(pr)

In [ ]:
pred_few['Normalized Prediction'] = nor_pre
pred_few['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,93
C,50
A,39
D,18


In [ ]:
pred_few['prompt'] = data2['prompt']
pred_few.head()

,Predicted,Normalized Prediction,prompt
0,د) 36,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,ب) الخلايا وحيدة النواة,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,ب) الشرايين,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,أ) 8,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,د) اصماغ,D,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_few.to_excel('Allam Few Shot QA Biology.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3846    0.4167    0.4000        36
           B     0.3656    0.5763    0.4474        59
           C     0.5000    0.4032    0.4464        62
           D     0.5000    0.2093    0.2951        43

    accuracy                         0.4150       200
   macro avg     0.4376    0.4014    0.3972       200
weighted avg     0.4396    0.4150    0.4058       200



# CoT

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=200,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
data3 = pd.read_excel('Biology-QA-ArabicPrompt-CoT.xlsx')

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3))):
    prompt = data3.iloc[i]["prompt"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("السؤال الذي يجب عليك الإجابة عليه:")[-1].strip()

    pred.append(answer)

100%|██████████| 200/200 [06:50<00:00,  2.05s/it]


In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
السؤال:\nإذا كان عدد جزيئات مستقبل الالكترونات النهائي في التنفس الخلوي 18 ، فما عدد جزيئات NADPH المستخدمة في حلقة كالفن :\n الخيارات:\nأ) 24\nب) 18\nج) 12\nد) 36 \nالإجابة:\nج) 12,1
السؤال:\nتتحول الخلايا ..... إلى خلايا بلعمية فى بعض الأحيان عند حاجة الجسم إليها\n الخيارات:\nأ) الخلايا التائية\nب) الخلايا وحيدة النواة\nج) الخلايا الحامضية\nد) الخلايا القاتلة الطبيعية \nالإجابة:\nب) الخلايا وحيدة النواة,1
السؤال:\nما اسم الوعاء الدموي الذي يحمل الدم المؤكسج بعيدا عن القلب؟\n الخيارات:\nأ) الصمام\nب) الشرايين\nج) الشعيرات\nد) الأوردة \nالإجابة:\nب) الشرايين,1
السؤال:\nخلية تحتوي على 16 كروموسوم في المرحلة النمو الأول فكم يكون عدد الكروموسومات في مرحلة النمو\n الخيارات:\nأ) 8\nب) 16\nج) 32\nد) 35 \nالإجابة:\nأ) 8,1
السؤال:\nكل المواد التالية وسائل مناعة تركيبية موجودة سلفا في النبات عدا\n الخيارات:\nأ) شموع\nب) شعيرات\nج) اشواك\nد) اصماغ \nالإجابة:\nأ) شموع,1
...,...
السؤال:\n تم تكوين حمض الستريك 6 مرات ، وهذا يعد دليلاً على أن عدد جزيئات الغلوكوز التي دخلت في عملية\n التنفس الخلوي الهوائي ...... جزيء :\n الخيارات:\nأ) 12\nب) 6\nج) 3\nد) 2 \nالإجابة:\nب) 6,1
السؤال:\nأى الخصائص التالية تجعل النبات أكثر دعامة فسيولوجية\n الخيارات:\nأ) عدد الثغور كبير وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية قليل\nب) عدد الثغور قليل وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية كبير\nج) عدد الثغور كبير وطبقة الكيوتين رقيقة وعدد الشعيرات الجذرية قليل\nد) عدد الثغور قليل وطبقة الكيوتين رقيقة وعدد الشعيرات كبير \nالإجابة:\nب) عدد الثغور قليل وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية كبير,1
السؤال:\nتتشابه عظام اليد الي مع عظام القدم فى\n الخيارات:\nأ) العدد\nب) الحجم\nج) التركيب\nد) أخر إجابتين \nالإجابة:\nج) التركيب,1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "الإجابة:" in pr:
    answer = pr.split("الإجابة:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  elif "الإجابة النهائية:" in pr:
    answer = pr.split("الإجابة النهائية:")[-1].strip()
    if "أ)" in answer:
      nor_pre.append("A")
    elif "أ / أ" in answer:
      nor_pre.append("A")
    elif "أ / ب / ج / د)" in answer:
      nor_pre.append("Unclassified")
    elif "ب)" in answer:
      nor_pre.append("B")
    elif "ب" in answer:
      nor_pre.append("B")
    elif "ج)" in answer:
      nor_pre.append("C")
    elif "ج" in answer:
      nor_pre.append("C")
    elif "د)" in answer:
      nor_pre.append("D")
  else:
    print(pr)

In [ ]:
pred_cot['Normalized Prediction'] = nor_pre
pred_cot['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
B,95
C,55
A,45
D,5


In [ ]:
pred_cot['prompt'] = data3['prompt']
pred_cot.head()

,Predicted,Normalized Prediction,prompt
0,السؤال:\nإذا كان عدد جزيئات مستقبل الالكترونات...,C,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
1,السؤال:\nتتحول الخلايا ..... إلى خلايا بلعمية ...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
2,السؤال:\nما اسم الوعاء الدموي الذي يحمل الدم ا...,B,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
3,السؤال:\nخلية تحتوي على 16 كروموسوم في المرحلة...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...
4,السؤال:\nكل المواد التالية وسائل مناعة تركيبية...,A,لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد ا...


In [ ]:
pred_cot.to_excel('Allam CoT QA Biology.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3556    0.4444    0.3951        36
           B     0.3895    0.6271    0.4805        59
           C     0.4545    0.4032    0.4274        62
           D     0.8000    0.0930    0.1667        43

    accuracy                         0.4100       200
   macro avg     0.4999    0.3920    0.3674       200
weighted avg     0.4918    0.4100    0.3812       200

